In [ ]:
# Cell 1: Import và Nạp dữ liệu ML-ready
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from utils.model_trainer import ModelTrainer

# Tự động reload code
%load_ext autoreload
%autoreload 2

# Nạp dữ liệu từ Notebook 06
df_ml = pd.read_parquet("../data/processed/MBB_ml_ready.parquet")
print("Kích thước dữ liệu nạp vào:", df_ml.shape)

# Cell 2: Khởi tạo Trainer và Chia tách Dữ liệu
trainer = ModelTrainer(df_ml, target_col='Target')
X_train, X_test, y_train, y_test = trainer.time_series_split(test_size=0.2)

# Cell 3: Huấn luyện Mô hình XGBoost với Tuning Mất cân bằng
import xgboost as xgb

# 1. Tính toán tỷ lệ mất cân bằng (Class Imbalance Ratio)
# Công thức: Số lượng Nhãn 0 (Đứng ngoài) / Số lượng Nhãn 1 (Mua)
num_class_0 = (y_train == 0).sum()
num_class_1 = (y_train == 1).sum()
pos_weight = num_class_0 / num_class_1

print(f"Phân bố tập Train - Nhãn 0: {num_class_0}, Nhãn 1: {num_class_1}")
print(f"🔄 Trọng số scale_pos_weight được tính toán: {pos_weight:.2f}")

# 2. Huấn luyện mô hình XGBoost (Truyền thêm tham số cân bằng)
model, metrics = trainer.train_xgboost(
    X_train, 
    y_train, 
    X_test, 
    y_test,
    scale_pos_weight=pos_weight,  # Bắt buộc AI phải chú ý vào lệnh MUA
    max_depth=5,                  # Khống chế độ sâu để tránh học vẹt (Overfitting)
    learning_rate=0.05            # Tốc độ học chậm lại để tìm điểm Mua chuẩn hơn
)

# 3. In kết quả đánh giá chi tiết
print("\n" + "="*50)
print("BÁO CÁO HIỆU SUẤT MÔ HÌNH TRÊN TẬP TEST")
print("="*50)
for metric_name, score in metrics.items():
    print(f"- {metric_name}: {score:.4f}")
print("="*50)

# Cell 4: Lưu mô hình đã huấn luyện (Dùng cho Backtest sau này)
import joblib
model_path = "../data/processed/xgboost_model.pkl"
joblib.dump(model, model_path)
print(f"\nĐã lưu mô hình tại: {model_path}")